In [23]:
import os
import pandas as pd
import openai
from openai import OpenAI, beta
import sys
import json
import pprint

sys.path.append(os.path.abspath(".."))
from src import config

from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
  api_key=os.getenv('OPENAI_API_KEY')
)

from docling.document_converter import DocumentConverter

## El problema que falta resolver es vincular las facturas con el comprobante previo que corresponda:
- arrendamientos: cuota arr cargada en albor
- fletes: con carta de porte
- labores: con OT
- gastos vehiculos: con patente del vehiculo
- gastos comerciales: tipo servicio (ej: sellado, comisión, etc.)

In [24]:
from pydantic import BaseModel
from typing import Optional

from pydantic import BaseModel
from typing import Optional, List

class InvoiceItem(BaseModel):
    codigo: str
    descripcion: str
    unidad_medida: str
    cantidad: float
    precio_unitario: float
    importe: float
    remito: Optional[str] = None
    mes_servicio: Optional[str] = None

class InvoiceTax(BaseModel):
    concepto: str
    base_imponible: Optional[float] = None
    impuesto: str
    monto_impuesto: Optional[float] = None
    total: float

class Invoice(BaseModel):
    tipo: str  # Factura Original o Duplicado
    numero: str
    fecha: str
    cuit_emisor: str
    dgr_inscripto: str
    inicio_actividad: str
    razon_social_cliente: str
    categoria_iva_cliente: str
    domicilio_cliente: str
    cuit_cliente: str
    codigo_cliente: str
    condicion_venta: str
    pedido: str
    vendedor: str
    deposito: str
    moneda: str
    tipo_cambio: float
    items: List[InvoiceItem]
    impuestos: List[InvoiceTax]
    subtotal: float
    total: float
    observaciones: Optional[str] = None
    cae: str
    vencimiento_cae: str
    codigo_barras: str

class Invoices(BaseModel):
    invoices: list[Invoice] 

In [25]:
# root_dir = '/Users/ju/Library/CloudStorage/OneDrive-ESPARTINAS.A/DocumentacionEspartina/PRODUCCION/Agricultura/Arrendamientos/25-26'
root_dir = '/Users/ju/Library/CloudStorage/OneDrive-ESPARTINAS.A/DocumentacionEspartina/TECNOLOGIA Y PROCESOS/PROYECTOS/_NO Vigentes/ERPs/Braulio - Ejemplos facturas'

In [26]:
file_list = []

for root, dirs, files in os.walk(root_dir):
    for file in files:
        if file.endswith('.pdf'):
            file_path = os.path.join(root, file)
            print(file_path)
            file_list.append(file_path)

/Users/ju/Library/CloudStorage/OneDrive-ESPARTINAS.A/DocumentacionEspartina/TECNOLOGIA Y PROCESOS/PROYECTOS/_NO Vigentes/ERPs/Braulio - Ejemplos facturas/2023 FA 2952.pdf
/Users/ju/Library/CloudStorage/OneDrive-ESPARTINAS.A/DocumentacionEspartina/TECNOLOGIA Y PROCESOS/PROYECTOS/_NO Vigentes/ERPs/Braulio - Ejemplos facturas/2023 FA 3696.pdf
/Users/ju/Library/CloudStorage/OneDrive-ESPARTINAS.A/DocumentacionEspartina/TECNOLOGIA Y PROCESOS/PROYECTOS/_NO Vigentes/ERPs/Braulio - Ejemplos facturas/2023-COM-1753.pdf
/Users/ju/Library/CloudStorage/OneDrive-ESPARTINAS.A/DocumentacionEspartina/TECNOLOGIA Y PROCESOS/PROYECTOS/_NO Vigentes/ERPs/Braulio - Ejemplos facturas/2023 COM 1687.pdf
/Users/ju/Library/CloudStorage/OneDrive-ESPARTINAS.A/DocumentacionEspartina/TECNOLOGIA Y PROCESOS/PROYECTOS/_NO Vigentes/ERPs/Braulio - Ejemplos facturas/2023 COM 596.pdf
/Users/ju/Library/CloudStorage/OneDrive-ESPARTINAS.A/DocumentacionEspartina/TECNOLOGIA Y PROCESOS/PROYECTOS/_NO Vigentes/ERPs/Braulio - Ejemplo

In [27]:
query = "Eres un prolijo y laborioso data entry, que extae muy detalladamente los datos de las facturas, para pasarlos a una tabla con el formato dado."

In [28]:
processed = []
dfs = []

for file_path in file_list:
    if file_path.endswith('.pdf'):
        print(f'Processing {file_path}')

        try:
            source = file_path # document per local path or URL
            converter = DocumentConverter()
            result = converter.convert(source)
            result_text = result.document.export_to_markdown()
        
        except Exception as e:
            print(f'Not processed: {file_path} - error: {e}')
            continue

        try:
            response = beta.chat.completions.parse(
                model="gpt-4o-mini-2024-07-18",
                messages=[
                    {
                        "role": "user",
                        "content": f"Los datos del archivo {file} están en formato markdown:\n{result_text}\n\nPregunta: {query}"
                    }
                ],
                temperature=0,
                max_tokens=15000,
                response_format=Invoice,
                top_p=1
            )

            # Extract the content from the response
            json_content = response.choices[0].message.content  # Accessing the attribute

            data = json.loads(json_content)

            print(data)

            # Convert to DataFrame
            df = pd.DataFrame(data)

            df['root'] = f'{root}'
            df['file'] = f'{file}'

            dfs.append(df)

            processed.append(file_path)

        except Exception as e:
            print(e)

if len(dfs) > 0:
    final_df = pd.concat(dfs)
else:
    final_df = pd.DataFrame()
    print('No dfs to concat')
    pass

Processing /Users/ju/Library/CloudStorage/OneDrive-ESPARTINAS.A/DocumentacionEspartina/TECNOLOGIA Y PROCESOS/PROYECTOS/_NO Vigentes/ERPs/Braulio - Ejemplos facturas/2023 FA 2952.pdf
{'tipo': 'Factura', 'numero': '0004-00050426', 'fecha': '31/07/2023', 'cuit_emisor': '30-70923071-5', 'dgr_inscripto': '30-70923071-5', 'inicio_actividad': '01/10/2005', 'razon_social_cliente': 'ESPARTINA S A', 'categoria_iva_cliente': 'RESPONSABLE INSCRIPTO', 'domicilio_cliente': 'ESTANCIA LOS ALAMOS, DAIREAUX, BUENOS AIRES', 'cuit_cliente': '30-63532214-0', 'codigo_cliente': '6555', 'condicion_venta': 'CUENTA CORRIENTE RESPONSABLE INSCRIPTO', 'pedido': '4-42615/4-42620/4-42632', 'vendedor': '', 'deposito': '', 'moneda': 'ARS', 'tipo_cambio': 1, 'items': [{'codigo': '3', 'descripcion': '(3)SUPER', 'unidad_medida': 'litros', 'cantidad': 4.6781, 'precio_unitario': 192.2, 'importe': 899.12, 'remito': None, 'mes_servicio': None}, {'codigo': '4', 'descripcion': '(4)V-POWER DIESEL', 'unidad_medida': 'litros', 'c

In [29]:
final_df

,tipo,numero,fecha,cuit_emisor,dgr_inscripto,inicio_actividad,razon_social_cliente,categoria_iva_cliente,domicilio_cliente,cuit_cliente,...,items,impuestos,subtotal,total,observaciones,cae,vencimiento_cae,codigo_barras,root,file
0,Factura,0004-00050426,31/07/2023,30-70923071-5,30-70923071-5,01/10/2005,ESPARTINA S A,RESPONSABLE INSCRIPTO,"ESTANCIA LOS ALAMOS, DAIREAUX, BUENOS AIRES",30-63532214-0,...,"{'codigo': '3', 'descripcion': '(3)SUPER', 'un...","{'concepto': 'IVA', 'base_imponible': 40458.13...",40458.13,51337.82,Es copia del original,,,,/Users/ju/Library/CloudStorage/OneDrive-ESPART...,2023 FA 3612.pdf
1,Factura,0004-00050426,31/07/2023,30-70923071-5,30-70923071-5,01/10/2005,ESPARTINA S A,RESPONSABLE INSCRIPTO,"ESTANCIA LOS ALAMOS, DAIREAUX, BUENOS AIRES",30-63532214-0,...,"{'codigo': '4', 'descripcion': '(4)V-POWER DIE...","{'concepto': 'Percepciones', 'base_imponible':...",40458.13,51337.82,Es copia del original,,,,/Users/ju/Library/CloudStorage/OneDrive-ESPART...,2023 FA 3612.pdf
0,Factura de Crédito,A00012-00000684,15/09/2023,30-66190442-5,901-154008-1,26/03/2004,ESPARTINA S.A,Responsable inscripto,MIÑONES 2239 - CABA,30-63532214-0,...,"{'codigo': 'NBLV', 'descripcion': 'Lenovo V15 ...","{'concepto': 'IVA', 'base_imponible': 10055.0,...",10055.00,11110.78,El pago deberá efectuarse en pesos según la co...,73375306090801,15/09/2023,306619044252010001273375306090801202309150,/Users/ju/Library/CloudStorage/OneDrive-ESPART...,2023 FA 3612.pdf
0,Factura,00000894,15/09/2023,20285715696,,01/05/2017,ESPARTINA S A,IVA Responsable Inscripto,"Estancia Los Alamos 0 - Daireaux, Buenos Aires",30635322140,...,"{'codigo': '001', 'descripcion': 'carnicería',...","{'concepto': 'IVA 21%', 'base_imponible': 4795...",55483.18,61812.40,None,73376319356399,25/09/2023,,/Users/ju/Library/CloudStorage/OneDrive-ESPART...,2023 FA 3612.pdf
1,Factura,00000894,15/09/2023,20285715696,,01/05/2017,ESPARTINA S A,IVA Responsable Inscripto,"Estancia Los Alamos 0 - Daireaux, Buenos Aires",30635322140,...,"{'codigo': '002', 'descripcion': 'artículos de...","{'concepto': 'IVA 10.5%', 'base_imponible': 50...",55483.18,61812.40,None,73376319356399,25/09/2023,,/Users/ju/Library/CloudStorage/OneDrive-ESPART...,2023 FA 3612.pdf


In [30]:
final_df.to_excel(f'{root_dir}/data_extraida.xlsx')